# 05 · Temporal probe

The linear probe (notebook 04) averages the 8 time slices of each clip, so it cannot see the order of the movement. Its main error was cutback confused with roller, two maneuvers that differ mostly in how the turn unfolds over time.

Here a small attention probe reads the 8 slices as a sequence. To isolate the effect of time order, the same architecture is trained twice:
- **ordered**: learned positional embeddings tag each slice with its position;
- **order-blind**: no positional embeddings, so the slices are an unordered set.

Each variant is trained with 5 seeds on the session-aware split (val for early stopping, test only for evaluation).

## 1. Setup and data

In [1]:
import sys
from pathlib import Path

# Make the project root importable (notebooks/ is one level below it)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

from src.data.features import load_features
from src.models.temporal_probe import TemporalProbe
from src.models.train import predict, train_probe

# Session-aware split and cached V-JEPA 2 features, in the same row order
splits = pd.read_csv(PROJECT_ROOT / "data" / "splits.csv")
X = load_features(splits)  # (n_clips, 8 time slices, 1024)

# Integer labels: class index = position in the sorted class list
classes = sorted(splits["maneuver"].unique())
y = splits["maneuver"].map({name: i for i, name in enumerate(classes)}).to_numpy()

# Boolean masks for each part of the split
is_train, is_val, is_test = (splits["split"] == s for s in ("train", "val", "test"))
X_train, y_train = X[is_train], y[is_train]
X_val, y_val = X[is_val], y[is_val]
X_test, y_test = X[is_test], y[is_test]

print("classes:", classes)
print("train", X_train.shape, "| val", X_val.shape, "| test", X_test.shape)
print("GPU:", torch.cuda.get_device_name(0))

classes: ['360', 'cutback-frontside', 'roller', 'take-off']
train (735, 8, 1024) | val (85, 8, 1024) | test (236, 8, 1024)
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


## 2. Ordered vs order-blind probe

In [2]:
SEEDS = range(5)
runs = []            # one row of metrics per (variant, seed)
test_preds = {}      # test predictions per (variant, seed), reused in section 4
ordered_models = []  # kept for the reversed-time check in section 3

for use_order in (True, False):
    variant = "ordered" if use_order else "order-blind"
    for seed in SEEDS:
        # Seed before building the model so the initial weights also depend on the seed
        torch.manual_seed(seed)
        model = TemporalProbe(num_classes=len(classes), use_order=use_order)
        model, val_f1 = train_probe(model, X_train, y_train, X_val, y_val, seed=seed)

        pred = predict(model, X_test)
        test_preds[(variant, seed)] = pred
        runs.append({
            "variant": variant, "seed": seed, "val_f1": val_f1,
            "test_acc": accuracy_score(y_test, pred),
            "test_f1": f1_score(y_test, pred, average="macro"),
        })
        if use_order:
            ordered_models.append(model)
        print(f"{variant:12s} seed {seed}: val F1 {val_f1:.3f} | test acc {runs[-1]['test_acc']:.3f} | test F1 {runs[-1]['test_f1']:.3f}")

runs = pd.DataFrame(runs)

# Mean ± std over seeds, next to the linear probe from notebook 04 (single deterministic run)
summary = runs.groupby("variant")[["test_acc", "test_f1"]].agg(["mean", "std"]).round(3)
summary.loc["linear probe (nb 04)"] = [0.852, 0.0, 0.856, 0.0]
summary

ordered      seed 0: val F1 0.989 | test acc 0.898 | test F1 0.895
ordered      seed 1: val F1 1.000 | test acc 0.941 | test F1 0.943
ordered      seed 2: val F1 0.989 | test acc 0.936 | test F1 0.933
ordered      seed 3: val F1 0.978 | test acc 0.886 | test F1 0.893
ordered      seed 4: val F1 1.000 | test acc 0.936 | test F1 0.941
order-blind  seed 0: val F1 0.978 | test acc 0.894 | test F1 0.901
order-blind  seed 1: val F1 0.968 | test acc 0.890 | test F1 0.891
order-blind  seed 2: val F1 0.978 | test acc 0.928 | test F1 0.933
order-blind  seed 3: val F1 0.979 | test acc 0.919 | test F1 0.922
order-blind  seed 4: val F1 0.978 | test acc 0.903 | test F1 0.911


test_acc        test_f1       
                         mean    std    mean    std
variant                                            
order-blind             0.907  0.016   0.911  0.017
ordered                 0.919  0.026   0.921  0.025
linear probe (nb 04)    0.852  0.000   0.856  0.000

## 3. Reversed time order

In [3]:
# Feed the test clips with their 8 slices in reverse order.
# .copy() is needed because torch cannot read numpy arrays with negative strides.
X_test_rev = X_test[:, ::-1].copy()

rows = []
for (variant, seed), pred in test_preds.items():
    model = ordered_models[seed] if variant == "ordered" else None
    if model is None:
        continue
    pred_rev = predict(model, X_test_rev)
    rows.append({
        "seed": seed,
        "test_f1": f1_score(y_test, pred, average="macro"),
        "reversed_f1": f1_score(y_test, pred_rev, average="macro"),
        "changed_preds": int((pred != pred_rev).sum()),  # clips whose prediction flips
    })

reversed_check = pd.DataFrame(rows)
print(reversed_check.round(3).to_string(index=False))
print(f"\nmean F1 drop when reversed: {(reversed_check.test_f1 - reversed_check.reversed_f1).mean():.3f}")

 seed  test_f1  reversed_f1  changed_preds
    0    0.895        0.855             40
    1    0.943        0.894             24
    2    0.933        0.850             39
    3    0.893        0.868             21
    4    0.941        0.884             32

mean F1 drop when reversed: 0.051


## 4. Per-class results

In [4]:
# Per-class F1 for each variant, averaged over the 5 seeds
per_class = {}
for variant in ("ordered", "order-blind"):
    scores = [f1_score(y_test, test_preds[(variant, s)], average=None) for s in SEEDS]
    per_class[variant] = np.mean(scores, axis=0)
per_class = pd.DataFrame(per_class, index=classes).round(3)
print("Per-class F1 (mean over seeds):")
print(per_class, "\n")

# Confusion matrix of the ordered probe, summed over the 5 seeds (each test clip counted 5 times)
cm = sum(confusion_matrix(y_test, test_preds[("ordered", s)], labels=range(len(classes))) for s in SEEDS)
cm = pd.DataFrame(cm, index=[f"true {c}" for c in classes], columns=[f"pred {c}" for c in classes])
print("Ordered probe, confusion summed over 5 seeds:")
print(cm)

# The error that motivated this notebook: cutback predicted as roller (linear probe: 11 of 47)
cut, rol = classes.index("cutback-frontside"), classes.index("roller")
cut_as_rol = cm.iloc[cut, rol] / len(SEEDS)
print(f"\ncutback -> roller, average per run: {cut_as_rol:.1f} of {(y_test == cut).sum()}")

Per-class F1 (mean over seeds):
                   ordered  order-blind
360                  0.923        0.902
cutback-frontside    0.873        0.844
roller               0.926        0.916
take-off             0.962        0.983 

Ordered probe, confusion summed over 5 seeds:
                        pred 360  pred cutback-frontside  pred roller  \
true 360                     264                       3           23   
true cutback-frontside         6                     188           39   
true roller                   11                       4          520   
true take-off                  0                       0            7   

                        pred take-off  
true 360                            0  
true cutback-frontside              2  
true roller                         0  
true take-off                     113  

cutback -> roller, average per run: 7.8 of 47


## 5. Findings

- **The attention probe beats the linear probe by about 6 points of macro F1** on the same frozen V-JEPA 2 features and the same session-aware split: 0.921 ± 0.025 (ordered) and 0.911 ± 0.017 (order-blind) over 5 seeds, against 0.856 for the linear probe. Even the worst seed (0.891) is above the linear probe.
- **Most of that gain comes from learned pooling, not from time order.** The ordered and order-blind probes differ by 1 point, which is smaller than the spread across seeds; seed by seed, the ordered probe wins 2, ties 1 and loses 2. The difference is not established.
- **The ordered probe does use slice order.** Feeding the 8 slices in reverse lowers its macro F1 by 0.051 on average, and 21 to 40 of the 236 test predictions change. Since the order-blind probe reaches similar accuracy without any order information, the order signal looks largely redundant: each V-JEPA 2 slice already covers a short window of motion.
- **Cutback is the class that benefits most from order**: F1 0.725 (linear) → 0.844 (order-blind) → 0.873 (ordered). Cutbacks predicted as roller fall from 11 of 47 (linear) to 7.8 of 47 per run (ordered), but this is still the main error. Take-off is slightly worse with order (0.983 → 0.962; 7 of 120 take-off predictions went to roller, summed over seeds). With 5 seeds, these per-class differences are indications, not tested effects.
- **Limitations.**
  - Validation macro F1 reached 0.97–1.00 in every run, so the 85-clip validation set barely discriminates between epochs during early stopping.
  - Reversing the slice order is not the same as reversing the video: the motion inside each slice still runs forward. The full reversed-clip experiment (re-encoding reversed videos with V-JEPA 2) is still planned.
  - All results come from a single split.
- **Next.** Repeat the comparison over several group-aware splits (group k-fold) to get confidence intervals, run the true reversed-video test, and check whether take-off relies on scene shortcuts (wave pools).